# Bovine Behavior — Exploratory Data Analysis

Dataset: Japanese Black Beef Cow (Zenodo 5849025)
Sensor: Tri-axial accelerometer, 25 Hz, neck-mounted


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
DATA_DIR = Path('../data')

## 1. Load data

In [ ]:
dfs = []
for i in range(1, 7):
    path = DATA_DIR / f'cow{i}.csv'
    if path.exists():
        df = pd.read_csv(path)
        df['cow_id'] = i
        dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print(data.shape)
data.head()

## 2. Basic info

In [ ]:
print(data.dtypes)
print('\nMissing values:')
print(data.isnull().sum())
print('\nBehavior classes:')
print(data['Label'].value_counts())

## 3. Class distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
data['Label'].value_counts().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Behavior class distribution')
ax.set_xlabel('Behavior')
ax.set_ylabel('Samples')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Accelerometer signal per behavior

In [ ]:
behaviors = data['Label'].unique()[:4]
fig, axes = plt.subplots(len(behaviors), 3, figsize=(14, 3 * len(behaviors)))

for i, beh in enumerate(behaviors):
    sample = data[data['Label'] == beh].head(250)
    for j, axis in enumerate(['AccX', 'AccY', 'AccZ']):
        axes[i, j].plot(sample[axis].values, linewidth=0.8)
        axes[i, j].set_title(f'{beh} — {axis}')
        axes[i, j].set_xlabel('Samples')
        axes[i, j].set_ylabel('g')

plt.tight_layout()
plt.show()

## 5. Feature engineering — sliding window

In [ ]:
def extract_features(df, window=125, step=62):
    features = []
    for start in range(0, len(df) - window, step):
        w = df.iloc[start:start + window]
        label = w['Label'].mode()[0]
        row = {'label': label, 'cow_id': w['cow_id'].iloc[0]}
        for ax in ['AccX', 'AccY', 'AccZ']:
            vals = w[ax].values
            row[f'{ax}_mean'] = vals.mean()
            row[f'{ax}_std'] = vals.std()
            row[f'{ax}_min'] = vals.min()
            row[f'{ax}_max'] = vals.max()
            row[f'{ax}_range'] = vals.max() - vals.min()
            fft = np.abs(np.fft.rfft(vals))
            row[f'{ax}_fft_mean'] = fft.mean()
            row[f'{ax}_fft_std'] = fft.std()
        features.append(row)
    return pd.DataFrame(features)

features_df = extract_features(data)
print(features_df.shape)
features_df.head()

## 6. PCA visualization

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = [c for c in features_df.columns if c not in ['label', 'cow_id']]
X = StandardScaler().fit_transform(features_df[feature_cols])

pca = PCA(n_components=2)
coords = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(10, 7))
for label in features_df['label'].unique():
    mask = features_df['label'] == label
    ax.scatter(coords[mask, 0], coords[mask, 1], label=label, alpha=0.5, s=15)
ax.set_title(f'PCA — Explained variance: {pca.explained_variance_ratio_.sum():.1%}')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()